In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import pandas as pd

from hawcsimulator.ali import ALISimulator

simulator = ALISimulator()

input = {
        "tangent_latitude": 30.0,
        "tangent_longitude": 0.0,
        "tangent_solar_zenith_angle": 60.0,
        "tangent_solar_azimuth_angle": 0.0,
        "time": pd.Timestamp("2022-01-01T12:00:00Z")
}

data = simulator.run(["l2", "l2_truth"], input, config={"atmosphere_method": "omps_calipso_era5"})

# The L2 granule holds one profile here: the first observation set's middle across-track bin.
# l2_truth is the true state on the same grid.  The retrieval fits measurements from
# lower_altitude_limit (5 km, or above any cloud) to 40 km
l2 = data["l2"].isel(observation_set=0, across_track=0)
truth = data["l2_truth"].isel(observation_set=0, across_track=0)
lower = float(l2["lower_altitude_limit"])
fitted = slice(lower, 40000)

plt.plot(l2["aerosol_extinction"].sel(altitude=fitted), l2["altitude"].sel(altitude=fitted), label="Retrieved")
plt.plot(truth["aerosol_extinction"].sel(altitude=fitted), truth["altitude"].sel(altitude=fitted), label="True")
plt.xlabel("Aerosol Extinction at 745 nm [/m]")
plt.ylabel("Altitude [m]")
plt.legend()